# Bureau of Labor Statistics Explorer

Advancing AI Anthropology through computational approaches to qualitative research.

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

<br>

---

<br>

## What This Notebook Does

This notebook retrieves labor and economic data from the Bureau of Labor Statistics (BLS) API. Access unemployment rates, consumer prices (CPI), employment statistics, wages, and productivity data by series ID or research preset. Results include time series data with optional calculations (percent changes, averages).

The BLS produces authoritative data on employment, wages, prices, and working conditions in the United States. This notebook provides structured access to that data for research without requiring knowledge of BLS series ID conventions.

## Key Features

1. **Research Presets**: Pre-built series groups for unemployment, CPI/inflation, employment, wages, and productivity
2. **Custom Series**: Enter any BLS series ID for specialized data
3. **Date Ranges**: Specify start and end years for historical analysis
4. **Calculations**: Optional percent changes and annual averages (API v2)
5. **Multi-Series**: Retrieve up to 50 series in one request
6. **Visualization**: Time series plots with branded styling
7. **Structured Export**: CSV, Excel, and JSON with metadata

## Workflow

1. **Setup**: Install packages and enter your free BLS API key
2. **Configure**: Select a preset or enter custom series IDs, set date range
3. **Retrieve**: Fetch data from the BLS API
4. **Review**: Browse results with time series charts
5. **Export**: Download structured data

## Applications

This notebook supports research that requires labor market and economic context — understanding employment patterns in fieldwork communities, tracking wage trends by industry, monitoring inflation's impact on household budgets, or contextualizing ethnographic observations within macroeconomic conditions.

## Methodological Positioning

This notebook represents a **computational anthropology** approach — using government labor statistics to complement qualitative research on work, economy, and livelihoods. BLS data captures formal labor market activity but not informal economies, unpaid labor, or subsistence activities.

**Important**: BLS series definitions, seasonal adjustments, and survey methodologies shape what the data measures. Consult BLS documentation for the specific series you use.

## Target Audience

Designed for anthropologists and qualitative researchers who need labor and economic context — from graduate students studying work and employment to applied researchers analyzing economic conditions in communities.

## Technical Approach

The notebook queries the BLS Public Data API v2 via HTTP POST requests. A free API key (available at [bls.gov/developers](https://www.bls.gov/developers/home.htm)) enables higher rate limits and additional features (calculations, annual averages). Without a key, the API v1 endpoint is used with reduced limits.

## AI Anthropology Toolkit

This notebook is part of the [AI Anthropology Toolkit](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit), a collection of computational notebooks for AI-assisted qualitative research. Alongside the toolkit's other structural-data explorers, it helps situate ethnography in structural context — the demographic, economic, and development conditions that ground a field site in its larger circumstances.

## License & Attribution

This notebook is released under the [PolyForm Noncommercial License 1.0.0](https://polyformproject.org/licenses/noncommercial/1.0.0): free for noncommercial use — research, education, and nonprofit work — with attribution to Matt Artz appreciated. For commercial licensing, contact [Matt Artz](https://www.mattartz.me/).

## Citation

If you use this toolkit in your academic research, please cite:

> Artz, Matt. 2026. AI Anthropology Toolkit. Software. Zenodo. https://doi.org/10.5281/zenodo.16728812

## References

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2026. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Anthropological Forum. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

## Setup and Installation

Install supporting libraries. Get a free BLS API key at [bls.gov/developers](https://www.bls.gov/developers/home.htm) — registration via the sign-up form. The API works without a key (v1) but with stricter limits (25 series, 10-year span).

In [ ]:
# Install required packages
!pip install -q requests pandas matplotlib openpyxl ipywidgets

import re
import json
import unicodedata
import requests
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

print("\u2713 All packages loaded")

IN_COLAB = False
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    pass

# --- BLS Series Presets ---

SERIES_PRESETS = {
    'Unemployment': {
        'description': 'National unemployment rates and labor force',
        'series': {
            'LNS14000000': 'Unemployment Rate (Seasonally Adjusted)',
            'LNS14000003': 'Unemployment Rate - White',
            'LNS14000006': 'Unemployment Rate - Black or African American',
            'LNS14000009': 'Unemployment Rate - Hispanic or Latino',
            'LNS14000012': 'Unemployment Rate - Asian',
            'LNS11000000': 'Civilian Labor Force Level',
            'LNS12000000': 'Employment Level',
            'LNS13000000': 'Unemployment Level',
        },
    },
    'Consumer Prices (CPI)': {
        'description': 'Consumer Price Index — inflation measures',
        'series': {
            'CUUR0000SA0': 'CPI-U All Items (Not Seasonally Adjusted)',
            'CUSR0000SA0': 'CPI-U All Items (Seasonally Adjusted)',
            'CUUR0000SAF1': 'CPI-U Food',
            'CUUR0000SAH1': 'CPI-U Shelter',
            'CUUR0000SAM': 'CPI-U Medical Care',
            'CUUR0000SAE1': 'CPI-U Energy',
            'CUUR0000SETB01': 'CPI-U Gasoline',
        },
    },
    'Employment by Industry': {
        'description': 'Current Employment Statistics (CES) — nonfarm payrolls',
        'series': {
            'CES0000000001': 'Total Nonfarm Employment',
            'CES0500000001': 'Total Private Employment',
            'CES1000000001': 'Mining and Logging',
            'CES2000000001': 'Construction',
            'CES3000000001': 'Manufacturing',
            'CES4000000001': 'Trade, Transportation, Utilities',
            'CES5000000001': 'Information',
            'CES6000000001': 'Financial Activities',
            'CES6500000001': 'Professional and Business Services',
            'CES7000000001': 'Education and Health Services',
            'CES8000000001': 'Leisure and Hospitality',
            'CES9000000001': 'Government',
        },
    },
    'Wages & Earnings': {
        'description': 'Average hourly and weekly earnings',
        'series': {
            'CES0500000003': 'Average Hourly Earnings - Total Private',
            'CES0500000011': 'Average Weekly Earnings - Total Private',
            'CES3000000003': 'Average Hourly Earnings - Manufacturing',
            'CES7000000003': 'Average Hourly Earnings - Education & Health',
            'CES8000000003': 'Average Hourly Earnings - Leisure & Hospitality',
        },
    },
    'Productivity': {
        'description': 'Labor productivity and costs',
        'series': {
            'PRS85006092': 'Nonfarm Business - Labor Productivity',
            'PRS85006112': 'Nonfarm Business - Unit Labor Costs',
            'PRS85006152': 'Nonfarm Business - Real Hourly Compensation',
            'PRS30006092': 'Manufacturing - Labor Productivity',
        },
    },
}

# --- Utilities ---

def make_slug(text):
    text = unicodedata.normalize('NFKD', text)
    text = re.sub(r'[^\w\s-]', '', text).strip().lower()
    return re.sub(r'[-\s]+', '_', text)[:50] or 'project'

def normalize_text(text):
    if not isinstance(text, str):
        return text
    for old, new in [('\u2011','-'),('\u2013','-'),('\u2014','-'),('\u2018',"'"),('\u2019',"'"),('\u201c','"'),('\u201d','"'),('\u2026','...')]:
        text = text.replace(old, new)
    return text

def style_excel(filepath):
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    wb = load_workbook(filepath)
    hf = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
    hfont = Font(bold=True, color='FFFFFF')
    tb = Border(left=Side(style='thin',color='E7ECEF'), right=Side(style='thin',color='E7ECEF'),
               top=Side(style='thin',color='E7ECEF'), bottom=Side(style='thin',color='E7ECEF'))
    for ws in wb.worksheets:
        for cell in ws[1]: cell.fill, cell.font, cell.alignment, cell.border = hf, hfont, Alignment(horizontal='center', vertical='center', wrap_text=True), tb
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(max(max(len(str(c.value or '')) for c in col)+2, 10), 60)
        ws.freeze_panes = 'A2'
        for row in ws.iter_rows(min_row=2):
            for cell in row: cell.alignment, cell.border = Alignment(vertical='top', wrap_text=True), tb
    wb.save(filepath)

print(f"\u2713 Environment: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"\u2713 {len(SERIES_PRESETS)} presets available")
print("\U0001f4c8 Ready to configure BLS data retrieval!")

## Configuration

Select a research preset or enter custom BLS series IDs. Set the date range for historical data. With an API key (v2), you get up to 50 series, 20-year spans, and percent change calculations.

In [ ]:
# Configuration

class BLSConfig:
    API_KEY = ''
    PRESET = 'Unemployment'
    CUSTOM_SERIES = ''
    START_YEAR = 2015
    END_YEAR = 2024
    CALCULATIONS = True
    ANNUAL_AVG = True
    PROJECT_NAME = ''

config = BLSConfig()

style = {'description_width': '160px'}
layout = widgets.Layout(width='500px')

current_year = datetime.now().year

def create_config_interface():

    config_html = '<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;">'
    config_html += '<h2 style="color: #274C77; margin-top: 0;">\U0001f4c8 Bureau of Labor Statistics</h2>'
    config_html += '<p><strong>Welcome!</strong> This notebook retrieves labor and economic data from the BLS API.</p>'
    config_html += '<h3 style="color: #274C77;">\U0001f4d6 How to Use:</h3>'
    config_html += '<ol>'
    config_html += '<li><strong>Configure:</strong> Set API key, preset, and date range below</li>'
    config_html += '<li><strong>Retrieve:</strong> Fetch time series data from BLS</li>'
    config_html += '<li><strong>Review:</strong> Browse data with charts</li>'
    config_html += '<li><strong>Export:</strong> Download structured results</li>'
    config_html += '</ol>'
    config_html += '</div>'

    instructions = widgets.HTML(value=config_html)

    api_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f511 API Key</h3>')

    api_key_input = widgets.Text(
        value='', placeholder='Enter your BLS API key (optional but recommended)',
        description='API Key:', layout=layout, style=style
    )

    api_help = widgets.HTML(
        '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
        'Get a free key at <a href="https://www.bls.gov/developers/home.htm" target="_blank">bls.gov/developers</a>. '
        'Without a key: max 25 series, 10-year span, no calculations. With a key: 50 series, 20 years, calculations included.</p>'
    )

    data_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\u2699\ufe0f Data Selection</h3>')

    preset_input = widgets.Dropdown(
        options=[(f'{k} \u2014 {v["description"]}', k) for k, v in SERIES_PRESETS.items()] + [('Custom Series IDs', 'custom')],
        value='Unemployment',
        description='Preset:', layout=layout, style=style
    )

    custom_series = widgets.Textarea(
        value='', placeholder='Enter BLS series IDs, one per line (e.g., LNS14000000)',
        description='Series IDs:',
        layout=widgets.Layout(width='500px', height='80px'), style=style
    )

    custom_help = widgets.HTML(
        '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
        'Find series IDs at <a href="https://data.bls.gov/dataQuery/find" target="_blank">data.bls.gov</a>. '
        'Common prefixes: LN (labor force), CU (CPI), CE (employment), PR (productivity).</p>'
    )

    custom_box = widgets.VBox([custom_series, custom_help])
    custom_box.layout.display = 'none'

    def on_preset_change(change):
        custom_box.layout.display = '' if change['new'] == 'custom' else 'none'
    preset_input.observe(on_preset_change, names='value')

    date_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4c5 Date Range</h3>')

    start_year = widgets.IntSlider(
        value=2015, min=1990, max=current_year,
        description='Start year:', style=style, layout=layout
    )

    end_year = widgets.IntSlider(
        value=current_year, min=1990, max=current_year,
        description='End year:', style=style, layout=layout
    )

    calc_toggle = widgets.Checkbox(
        value=True, description='Include percent changes (requires API key)',
        style={'description_width': 'auto'}
    )

    avg_toggle = widgets.Checkbox(
        value=True, description='Include annual averages (requires API key)',
        style={'description_width': 'auto'}
    )

    project_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4cb Project</h3>')

    project_name = widgets.Text(
        value='', placeholder='e.g., Labor Market Context',
        description='Project name:', layout=layout, style=style
    )

    save_btn = widgets.Button(
        description='Save Configuration',
        style={'button_color': '#6096BA'},
        layout=widgets.Layout(width='200px', margin='20px 0')
    )

    status = widgets.Output()

    def save_config(btn):
        config.API_KEY = api_key_input.value.strip()
        config.PRESET = preset_input.value
        config.CUSTOM_SERIES = custom_series.value.strip()
        config.START_YEAR = start_year.value
        config.END_YEAR = end_year.value
        config.CALCULATIONS = calc_toggle.value
        config.ANNUAL_AVG = avg_toggle.value
        config.PROJECT_NAME = project_name.value.strip()

        with status:
            clear_output()

            if config.START_YEAR > config.END_YEAR:
                print("\u26a0\ufe0f Start year must be before end year.")
                return

            span = config.END_YEAR - config.START_YEAR
            max_span = 20 if config.API_KEY else 10
            if span > max_span:
                print(f"\u26a0\ufe0f Date range ({span} years) exceeds {'v2' if config.API_KEY else 'v1'} limit of {max_span} years.")
                return

            if config.PRESET == 'custom':
                series_count = len([s.strip() for s in config.CUSTOM_SERIES.split('\n') if s.strip()])
                print(f"\u2713 Custom series: {series_count} IDs")
            else:
                series_count = len(SERIES_PRESETS[config.PRESET]['series'])
                print(f"\u2713 Preset: {config.PRESET} ({series_count} series)")

            print(f"\u2713 Date range: {config.START_YEAR}\u2013{config.END_YEAR}")
            if config.API_KEY:
                print(f"\u2713 API key: provided (v2 \u2014 calculations and averages available)")
            else:
                print("\u26a0\ufe0f No API key \u2014 using v1 (limited features)")
            print()
            print("\u2713 Configuration saved! Proceed to Retrieve.")

    save_btn.on_click(save_config)

    display(widgets.VBox([
        instructions,
        api_header, api_key_input, api_help,
        data_header, preset_input, custom_box,
        date_header, start_year, end_year, calc_toggle, avg_toggle,
        project_header, project_name,
        save_btn, status,
    ]))

create_config_interface()

## Retrieve Data

Fetch time series data from the BLS API. Each series returns monthly or quarterly observations with values and period labels.

In [ ]:
# Retrieve Data

bls_df = None
series_labels = {}

def fetch_bls_data():
    """Fetch data from BLS API v1 or v2."""
    # Determine series IDs
    if config.PRESET == 'custom':
        series_ids = [s.strip() for s in config.CUSTOM_SERIES.split('\n') if s.strip()]
        labels = {s: s for s in series_ids}
    else:
        preset = SERIES_PRESETS[config.PRESET]
        series_ids = list(preset['series'].keys())
        labels = preset['series']

    # Build request
    if config.API_KEY:
        url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'
        payload = {
            'seriesid': series_ids,
            'startyear': str(config.START_YEAR),
            'endyear': str(config.END_YEAR),
            'registrationkey': config.API_KEY,
            'calculations': config.CALCULATIONS,
            'annualaverage': config.ANNUAL_AVG,
        }
    else:
        url = 'https://api.bls.gov/publicAPI/v1/timeseries/data/'
        payload = {
            'seriesid': series_ids,
            'startyear': str(config.START_YEAR),
            'endyear': str(config.END_YEAR),
        }

    headers = {'Content-type': 'application/json'}
    response = requests.post(url, data=json.dumps(payload), headers=headers, timeout=30)

    if response.status_code != 200:
        raise RuntimeError(f"API returned status {response.status_code}")

    data = response.json()

    if data.get('status') != 'REQUEST_SUCCEEDED':
        msg = '; '.join(data.get('message', ['Unknown error']))
        raise RuntimeError(f"BLS API error: {msg}")

    return data['Results']['series'], labels


retrieve_btn = widgets.Button(
    description='Retrieve Data',
    style={'button_color': '#6096BA'},
    layout=widgets.Layout(width='200px', margin='10px 0')
)

progress_html = widgets.HTML(value='')
retrieve_out = widgets.Output()

def on_retrieve(_):
    global bls_df, series_labels
    retrieve_out.clear_output()
    bls_df = None

    progress_html.value = '<span style="color: #274C77;">Querying BLS API...</span>'

    with retrieve_out:
        print(f"\U0001f4c8 Fetching {config.PRESET if config.PRESET != 'custom' else 'custom'} data...")
        print(f"   Date range: {config.START_YEAR}\u2013{config.END_YEAR}")
        print()

    try:
        series_data, labels = fetch_bls_data()
        series_labels = labels
        progress_html.value = ''

        # Parse into rows
        all_rows = []
        for series in series_data:
            sid = series['seriesID']
            label = labels.get(sid, sid)

            for obs in series.get('data', []):
                row = {
                    'series_id': sid,
                    'series_name': label,
                    'year': int(obs['year']),
                    'period': obs['period'],
                    'period_name': obs.get('periodName', ''),
                    'value': float(obs['value']) if obs['value'] != '-' else None,
                }

                # Add calculations if available (v2)
                calcs = obs.get('calculations', {})
                if calcs:
                    pct = calcs.get('pct_changes', {})
                    for pct_key, pct_val in pct.items():
                        row[f'pct_change_{pct_key}'] = float(pct_val) if pct_val else None

                all_rows.append(row)

        if not all_rows:
            with retrieve_out:
                print("\u26a0\ufe0f No data returned.")
            return

        bls_df = pd.DataFrame(all_rows)
        bls_df.sort_values(['series_name', 'year', 'period'], inplace=True)
        bls_df.reset_index(drop=True, inplace=True)

        with retrieve_out:
            n_series = bls_df['series_id'].nunique()
            print(f"\u2713 Retrieved {len(bls_df)} observations across {n_series} series")
            print()

            for sid in bls_df['series_id'].unique():
                subset = bls_df[bls_df['series_id'] == sid]
                label = labels.get(sid, sid)
                latest = subset.iloc[-1]
                print(f"   {label}: {subset['value'].dropna().iloc[-1] if not subset['value'].dropna().empty else 'N/A'} "
                      f"(latest: {latest['period_name']} {latest['year']})")

    except Exception as e:
        progress_html.value = ''
        with retrieve_out:
            print(f"\u274c Error: {type(e).__name__}: {e}")

retrieve_btn.on_click(on_retrieve)
display(widgets.VBox([retrieve_btn, progress_html, retrieve_out]))

## Review Data

Browse the retrieved data with time series charts and summary statistics.

In [ ]:
# Review Data

COLORS = ['#274C77', '#6096BA', '#A3CEF1', '#8B8C89', '#4A7C59', '#CC6644', '#7B5EA7', '#C9A227']

if bls_df is None:
    print("\u26a0\ufe0f Run the Retrieve cell first.")
else:
    # Time series chart
    series_names = bls_df['series_name'].unique()

    fig, ax = plt.subplots(figsize=(12, 6))
    fig.patch.set_facecolor('#FFFFFF')
    ax.set_facecolor('#FAFAFA')

    for i, name in enumerate(series_names):
        subset = bls_df[(bls_df['series_name'] == name) & (bls_df['period'] != 'M13')]  # Exclude annual avg
        if subset.empty:
            continue
        # Build date-like x axis from year + period
        x_labels = [f"{row['year']}-{row['period'][1:]}" for _, row in subset.iterrows()]
        color = COLORS[i % len(COLORS)]
        ax.plot(range(len(x_labels)), subset['value'].values, label=name[:40], color=color, linewidth=1.5)

    # Show only every Nth x tick
    n_ticks = min(20, len(x_labels))
    step = max(1, len(x_labels) // n_ticks)
    ax.set_xticks(range(0, len(x_labels), step))
    ax.set_xticklabels([x_labels[i] for i in range(0, len(x_labels), step)], rotation=45, ha='right', fontsize=8)

    ax.set_title(f'BLS Data \u2014 {config.PRESET}', fontsize=14, color='#274C77', fontweight='bold', pad=15)
    ax.set_ylabel('Value', fontsize=11, color='#274C77')
    ax.legend(frameon=True, facecolor='white', edgecolor='#E7ECEF', fontsize=8, loc='best')
    ax.grid(True, alpha=0.3, color='#8B8C89')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#E7ECEF')
    ax.spines['bottom'].set_color('#E7ECEF')

    plt.tight_layout()
    plt.show()

    # Summary table
    print()
    print("\U0001f4cb Series Summary:")
    for name in series_names:
        subset = bls_df[(bls_df['series_name'] == name) & (bls_df['period'] != 'M13')]
        vals = subset['value'].dropna()
        if vals.empty:
            continue
        print(f"\n   {name}:")
        print(f"      Latest: {vals.iloc[-1]}  |  Min: {vals.min()}  |  Max: {vals.max()}  |  Mean: {vals.mean():.1f}")

## Export

Export BLS data as CSV, Excel, or JSON.

In [ ]:
# Export

if bls_df is None:
    print("\u26a0\ufe0f Run the Retrieve cell first.")
else:
    export_format = widgets.Dropdown(
        options=[
            ('CSV (.csv)', 'csv'),
            ('Excel (.xlsx) \u2014 styled with metadata', 'excel'),
            ('JSON (.json) \u2014 full data with query metadata', 'json'),
        ],
        value='excel',
        description='Format:',
        style={'description_width': '80px'},
        layout=widgets.Layout(width='500px')
    )

    export_btn = widgets.Button(
        description='Export',
        style={'button_color': '#6096BA'},
        layout=widgets.Layout(width='200px', margin='10px 0')
    )

    export_out = widgets.Output()

    def on_export(_):
        export_out.clear_output()
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        slug = make_slug(config.PROJECT_NAME) if config.PROJECT_NAME else 'bls'
        preset_slug = make_slug(config.PRESET)[:15]
        fmt = export_format.value

        try:
            if fmt == 'csv':
                filepath = f'{slug}_{preset_slug}_{timestamp}.csv'
                bls_df.to_csv(filepath, index=False)
                with export_out:
                    print(f"\u2713 {filepath}")
                    if IN_COLAB:
                        colab_files.download(filepath)

            elif fmt == 'excel':
                filepath = f'{slug}_{preset_slug}_{timestamp}.xlsx'
                with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
                    bls_df.to_excel(writer, sheet_name='BLS Data', index=False)
                    meta_rows = [
                        {'Field': 'Preset', 'Value': config.PRESET},
                        {'Field': 'Date Range', 'Value': f'{config.START_YEAR}-{config.END_YEAR}'},
                        {'Field': 'Series Count', 'Value': bls_df['series_id'].nunique()},
                        {'Field': 'Observations', 'Value': len(bls_df)},
                        {'Field': 'API Version', 'Value': 'v2' if config.API_KEY else 'v1'},
                        {'Field': 'Retrieved', 'Value': datetime.now().isoformat()},
                    ]
                    if config.PROJECT_NAME:
                        meta_rows.insert(0, {'Field': 'Project', 'Value': config.PROJECT_NAME})
                    pd.DataFrame(meta_rows).to_excel(writer, sheet_name='Metadata', index=False)
                style_excel(filepath)
                with export_out:
                    print(f"\u2713 {filepath}")
                    if IN_COLAB:
                        colab_files.download(filepath)

            elif fmt == 'json':
                filepath = f'{slug}_{preset_slug}_{timestamp}.json'
                output = {
                    'metadata': {
                        'project_name': config.PROJECT_NAME,
                        'preset': config.PRESET,
                        'start_year': config.START_YEAR,
                        'end_year': config.END_YEAR,
                        'n_series': bls_df['series_id'].nunique(),
                        'n_observations': len(bls_df),
                        'retrieved_at': datetime.now().isoformat(),
                    },
                    'series_labels': series_labels,
                    'data': json.loads(bls_df.to_json(orient='records')),
                }
                with open(filepath, 'w') as f:
                    json.dump(output, f, indent=2, default=str)
                with export_out:
                    print(f"\u2713 {filepath}")
                    if IN_COLAB:
                        colab_files.download(filepath)

        except Exception as e:
            with export_out:
                print(f"\u274c Export error: {type(e).__name__}: {e}")

    export_btn.on_click(on_export)
    display(widgets.VBox([export_format, export_btn, export_out]))